# NB_patient_360 - Silver to Gold

Builds `<catalog>.gold.patient_360` (one row per patient) per the STTM `sttm_silver_to_gold.xlsx`. Driving table `silver.patients` LEFT JOINed to `hospitals`, `insurance_providers` and aggregated `lab_results`. Load type: FULL (CREATE OR REPLACE TABLE ... AS SELECT).

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_hospital.gold;

CREATE OR REPLACE TABLE adb_hospital.gold.patient_360 AS

WITH lab_agg AS (
    SELECT
        patient_id,
        COUNT(*) AS total_tests,
        COUNT_IF(result_status = 'Abnormal') AS abnormal_test_count,
        COUNT_IF(result_status = 'Critical') AS critical_test_count,
        MAX(test_date) AS last_test_date
    FROM adb_hospital.silver.lab_results
    GROUP BY patient_id
)

SELECT
    p.patient_id,
    p.first_name,
    p.last_name,
    CAST(p.date_of_birth AS DATE) AS date_of_birth,
    CAST(FLOOR(DATEDIFF(CURRENT_DATE(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
    p.gender,
    p.city,
    p.state,
    p.registered_hospital_id AS hospital_id,
    h.hospital_name,
    p.insurance_provider_id,
    ip.provider_name AS insurance_provider_name,
    COALESCE(l.total_tests, 0) AS total_tests,
    COALESCE(l.abnormal_test_count, 0) AS abnormal_test_count,
    COALESCE(l.critical_test_count, 0) AS critical_test_count,
    l.last_test_date

FROM adb_hospital.silver.patients p

LEFT JOIN adb_hospital.silver.hospitals h
    ON p.registered_hospital_id = h.hospital_id

LEFT JOIN adb_hospital.silver.insurance_providers ip
    ON p.insurance_provider_id = ip.insurance_provider_id

LEFT JOIN lab_agg l
    ON p.patient_id = l.patient_id;

In [0]:
%sql
select * from adb_hospital.gold.patient_360;

In [0]:
total_count = spark.table("adb_hospital.gold.patient_360").count()
print(f"patient_360 total count: {total_count}")
dbutils.notebook.exit(str(total_count))